# Notebook 1 — Data Engineering & Merging
### Pakistan Export Demand Forecasting System | Final Year Project

---

## Purpose
Transform **13 raw CSV files** into a single, model-ready **`Master_FYP_Dataset.csv`** — the single source of truth for all downstream notebooks.

## Pipeline Overview

| Step | Task |
|------|------|
| 1 | Load 10 commodity CSVs + 3 external driver CSVs |
| 2 | Fix HS 7403 (Copper) — 6 missing months filled with `0` (no trade) |
| 3 | Drop `Net_Weight_KG`, validate and clean `Export_Value_USD` |
| 4 | Stack all 10 commodities **row-wise** → panel (long) format |
| 5 | Merge external drivers **column-wise** on `Date_YYYYMM` |
| 6 | Engineer lag, rolling-average, and seasonal features |
| 7 | Drop NaN rows from early lag period (first 6 months per commodity) |
| 8 | Save `Master_FYP_Dataset.csv` |

**Expected final shape:** `1,860 rows × 16 columns`  
*(10 commodities × 186 months, after dropping the first 6 NaN-lag months per commodity)*

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('Libraries loaded successfully.')

Libraries loaded successfully.


---
## Section 1 — Configuration & File Paths

In [8]:
# All paths are relative to this notebook inside /Notebooks/
BASE_DIR     = Path('..')
COMTRADE_DIR = BASE_DIR / 'Data' / 'Uncomtrade' / 'training_data' / 'uncomtrade'
EXTERNAL_DIR = BASE_DIR / 'Data' / 'Uncomtrade' / 'training_data' / 'external_drives'
OUTPUT_DIR   = BASE_DIR / 'Data'

# Complete monthly timeline: January 2010 to December 2025 (192 months)
DATE_SPINE = [
    int(f'{year}{month:02d}')
    for year in range(2010, 2026)
    for month in range(1, 13)
]

# Commodity file registry — HS Code : CSV filename
COMMODITY_FILES = {
    1006: 'FYP_Export_Data_1006_Rice_2010_2025.csv',
    1207: 'FYP_Export_Data_1207_Oil Seeds_2010_2025.csv',
    2523: 'FYP_Export_Data_2523_Cement_2010_2025.csv',
    5205: 'FYP_Export_Data_5205 _Cotton_Yarn_2010_2025_1_miss.csv',
    6110: 'FYP_Export_Data_6110 _Jerseys_Pullovers_Cardigans_2010_2025_1_record_miss.csv',
    6203: 'FYP_Export_Data_6203___Men_Suits_Jackets_Trousers_2010_2025.csv',
    6302: 'FYP_Export_Data_6302_Bed_Table_Toilet_Kitchen_Linens_2010_2025.csv',
    7403: 'FYP_Export_Data_7403_Refined Copper_2010_2025.csv',
    9018: 'FYP_Export_Data_9018_Medical_Surgical_Instruments_2010_2025.csv',
    9506: 'FYP_Export_Data_9506_Sports_Goods_Inflatable_Balls_2010_2025.csv',
}

# External driver file registry — target column name : CSV filename
EXTERNAL_FILES = {
    'USD_PKR_Close'        : '1_USD_PKR_Monthly.csv',
    'Brent_Oil_Avg'        : '2_Brent_Oil_Monthly.csv',
    'US_Consumer_Confidence': 'US_Consumer_Confidence_2010_2025.csv',
}

print(f'Commodity files   : {len(COMMODITY_FILES)}')
print(f'External drivers  : {len(EXTERNAL_FILES)}')
print(f'Date spine        : {len(DATE_SPINE)} months  ({DATE_SPINE[0]} to {DATE_SPINE[-1]})')
print(f'Expected rows     : {len(COMMODITY_FILES) * len(DATE_SPINE):,}  (before NaN drop)')

Commodity files   : 10
External drivers  : 3
Date spine        : 192 months  (201001 to 202512)
Expected rows     : 1,920  (before NaN drop)


---
## Section 2 — Load Raw Data

In [9]:
raw_commodity_dfs = {}

for hs_code, filename in COMMODITY_FILES.items():
    filepath = COMTRADE_DIR / filename
    df = pd.read_csv(filepath, dtype={'Date_YYYYMM': int})
    raw_commodity_dfs[hs_code] = df

print('{:<10} {:<8} {:<18} {:<12} {}'.format('HS Code', 'Rows', 'Missing Months', 'Null USD', 'Status'))
print('-' * 65)
for hs_code, df in raw_commodity_dfs.items():
    null_usd       = df['Export_Value_USD'].isnull().sum()
    missing_months = len(DATE_SPINE) - len(df)
    status         = '<-- NEEDS FIX' if missing_months > 0 else 'OK'
    print(f'{hs_code:<10} {len(df):<8} {missing_months:<18} {null_usd:<12} {status}')

HS Code    Rows     Missing Months     Null USD     Status
-----------------------------------------------------------------
1006       192      0                  0            OK
1207       192      0                  0            OK
2523       192      0                  0            OK
5205       192      0                  0            OK
6110       192      0                  0            OK
6203       192      0                  0            OK
6302       192      0                  0            OK
7403       186      6                  0            <-- NEEDS FIX
9018       192      0                  0            OK
9506       192      0                  0            OK


In [10]:
raw_external_dfs = {}

for col_name, filename in EXTERNAL_FILES.items():
    filepath = EXTERNAL_DIR / filename
    df = pd.read_csv(filepath, dtype={'Date_YYYYMM': int})
    raw_external_dfs[col_name] = df

print('{:<28} {:<8} {:<26} {}'.format('Driver', 'Rows', 'Range', 'Nulls'))
print('-' * 70)
for col_name, df in raw_external_dfs.items():
    data_col = df.columns[1]
    nulls    = df[data_col].isnull().sum()
    rng      = f"{df['Date_YYYYMM'].min()} to {df['Date_YYYYMM'].max()}"
    print(f'{col_name:<28} {len(df):<8} {rng:<26} {nulls}')

Driver                       Rows     Range                      Nulls
----------------------------------------------------------------------
USD_PKR_Close                192      201001 to 202512           0
Brent_Oil_Avg                192      201001 to 202512           0
US_Consumer_Confidence       192      201001 to 202512           0


---
## Section 3 — Fix HS 7403 (Refined Copper): Fill 6 Missing Months

UN Comtrade simply **omits months where Pakistan reported zero trade** — the rows don't exist in the raw file.  
We build a complete 192-month date spine, left-merge the copper data onto it, and fill the gaps with `Export_Value_USD = 0.0` (confirmed no-trade months, not missing data).

In [11]:
copper_raw  = raw_commodity_dfs[7403].copy()
spine_df    = pd.DataFrame({'Date_YYYYMM': DATE_SPINE})

copper_full = spine_df.merge(copper_raw, on='Date_YYYYMM', how='left')

# Show exactly which months were missing from the raw source
missing_months = copper_full.loc[copper_full['Export_Value_USD'].isnull(), 'Date_YYYYMM'].tolist()
print(f'Missing months detected ({len(missing_months)}): {missing_months}')

# Fill static metadata for the inserted (previously absent) rows
copper_full['Reporter']       = 'Pakistan'
copper_full['Partner']        = 'World'
copper_full['HS_Code']        = 7403
copper_full['Commodity_Name'] = copper_raw['Commodity_Name'].iloc[0]

# Zero-fill the target — this is genuine no-trade data, not missing observation
copper_full['Export_Value_USD'] = copper_full['Export_Value_USD'].fillna(0.0)

print(f'\nRows before fix  : {len(copper_raw)}')
print(f'Rows after fix   : {len(copper_full)}')
print(f'Rows added       : {len(copper_full) - len(copper_raw)}')
print(f'Null check       : {copper_full["Export_Value_USD"].isnull().sum()} nulls remaining')

# Update registry with the fixed version
raw_commodity_dfs[7403] = copper_full

Missing months detected (6): [201006, 201512, 201603, 201606, 201607, 201610]

Rows before fix  : 186
Rows after fix   : 192
Rows added       : 6
Null check       : 0 nulls remaining


---
## Section 4 — Standardize & Clean All Commodity Files

For every commodity file:
- **Drop `Net_Weight_KG`** — missing for ~50% of commodities by design (apparel and medical instruments are measured in piece-counts, not weight). Dropping ensures 100% completeness on the target.
- **Coerce `Export_Value_USD` to numeric** — catches any formatting artifacts (e.g., comma-separated strings).
- **Fill residual nulls in the target with `0`** — any remaining null means no trade that month.

In [12]:
KEEP_COLS = ['Date_YYYYMM', 'Reporter', 'Partner', 'HS_Code', 'Commodity_Name', 'Export_Value_USD']

cleaned_dfs = {}

for hs_code, df in raw_commodity_dfs.items():
    clean = df[KEEP_COLS].copy()

    # Coerce to numeric — handles comma-formatted strings or other source artifacts
    clean['Export_Value_USD'] = pd.to_numeric(clean['Export_Value_USD'], errors='coerce')

    null_count = clean['Export_Value_USD'].isnull().sum()
    if null_count > 0:
        print(f'HS {hs_code}: {null_count} null(s) in Export_Value_USD -- filled with 0')
        clean['Export_Value_USD'] = clean['Export_Value_USD'].fillna(0.0)

    clean['HS_Code']     = hs_code
    clean['Date_YYYYMM'] = clean['Date_YYYYMM'].astype(int)

    cleaned_dfs[hs_code] = clean

print('All commodity files standardized.')
print(f'Net_Weight_KG dropped. Columns retained: {KEEP_COLS}')

All commodity files standardized.
Net_Weight_KG dropped. Columns retained: ['Date_YYYYMM', 'Reporter', 'Partner', 'HS_Code', 'Commodity_Name', 'Export_Value_USD']


---
## Section 5 — Stack Row-Wise into Panel (Long) Format

All 10 cleaned commodity DataFrames are **concatenated vertically** using `pd.concat`.  
The result is a **panel dataset** where each row represents one commodity in one month.

> Sorted by `[HS_Code, Date_YYYYMM]` immediately — this ordering is **required** for the grouped lag and rolling computations in Section 7.

In [13]:
panel_df = pd.concat(cleaned_dfs.values(), ignore_index=True)

# Sort by commodity first, then time — critical for correct grouped feature engineering
panel_df = panel_df.sort_values(['HS_Code', 'Date_YYYYMM']).reset_index(drop=True)

expected_rows = len(COMMODITY_FILES) * len(DATE_SPINE)

print(f'Expected rows      : {expected_rows:,}')
print(f'Actual rows        : {len(panel_df):,}  -- Match: {len(panel_df) == expected_rows}')
print(f'Shape              : {panel_df.shape}')
print(f'Date range         : {panel_df["Date_YYYYMM"].min()}  to  {panel_df["Date_YYYYMM"].max()}')
print(f'Total nulls (USD)  : {panel_df["Export_Value_USD"].isnull().sum()}')
print()
print('Rows per commodity:')
print(panel_df.groupby('HS_Code').size().rename('Row Count').to_string())

Expected rows      : 1,920
Actual rows        : 1,920  -- Match: True
Shape              : (1920, 6)
Date range         : 201001  to  202512
Total nulls (USD)  : 0

Rows per commodity:
HS_Code
1006    192
1207    192
2523    192
5205    192
6110    192
6203    192
6302    192
7403    192
9018    192
9506    192


---
## Section 6 — Merge External Drivers (Column-Wise)

The three macroeconomic drivers are joined onto the panel using a **left join on `Date_YYYYMM`**.  
Since each driver has one value per month (not per commodity), the same driver values are **broadcast across all 10 commodity rows** for that month — which is the correct behaviour for global economic signals.

In [14]:
# Combine all three external drivers into one DataFrame keyed on Date_YYYYMM
ext_df = pd.DataFrame({'Date_YYYYMM': DATE_SPINE})

for col_name, driver_df in raw_external_dfs.items():
    data_col = driver_df.columns[1]   # second column is the driver value
    temp     = driver_df[['Date_YYYYMM', data_col]].rename(columns={data_col: col_name})
    ext_df   = ext_df.merge(temp, on='Date_YYYYMM', how='left')

print(f'External drivers shape : {ext_df.shape}')
print(f'Nulls in drivers       : {ext_df.isnull().sum().sum()}')
print()

# Left-join onto the panel — drivers broadcast to all 10 commodity rows per month
master_df = panel_df.merge(ext_df, on='Date_YYYYMM', how='left')

DRIVER_COLS = list(EXTERNAL_FILES.keys())
print(f'Shape after merge      : {master_df.shape}')
print(f'Null check (drivers)   : {master_df[DRIVER_COLS].isnull().sum().sum()} nulls')
print()
print('Sample — first 3 months across 2 commodities:')
sample = master_df[master_df['HS_Code'].isin([1006, 7403])].head(6)
print(sample[['Date_YYYYMM', 'HS_Code', 'Export_Value_USD'] + DRIVER_COLS].to_string(index=False))

External drivers shape : (192, 4)
Nulls in drivers       : 0

Shape after merge      : (1920, 9)
Null check (drivers)   : 0 nulls

Sample — first 3 months across 2 commodities:
 Date_YYYYMM  HS_Code  Export_Value_USD  USD_PKR_Close  Brent_Oil_Avg  US_Consumer_Confidence
      201001     1006    235,850,504.30          83.75          77.01                   74.40
      201002     1006    206,046,142.98          85.00          74.91                   73.60
      201003     1006    256,035,371.79          83.75          79.93                   73.60
      201004     1006    189,097,493.31          83.90          85.75                   72.20
      201005     1006    170,885,895.13          83.97          76.66                   73.60
      201006     1006    200,437,802.51          85.45          75.55                   76.00


---
## Section 7 — Feature Engineering

### Features Created

| Feature | Type | Description |
|---------|------|-------------|
| `Year` | Temporal | Calendar year (integer) |
| `Month` | Temporal | Calendar month 1–12 (integer) |
| `Month_Sin` | Seasonal | `sin(2π × month / 12)` — cyclic encoding |
| `Month_Cos` | Seasonal | `cos(2π × month / 12)` — cyclic encoding |
| `Lag_1M` | Lag | Export value 1 month prior |
| `Lag_3M` | Lag | Export value 3 months prior |
| `Lag_6M` | Lag | Export value 6 months prior |
| `Rolling_3M_Avg` | Rolling | Trailing 3-month average (no current-month leakage) |
| `Rolling_6M_Avg` | Rolling | Trailing 6-month average (no current-month leakage) |

> **Critical design choice:** All lag and rolling features are computed **grouped by `HS_Code`**.  
> Without grouping, the lag for January 2010 of Rice would incorrectly pull December 2025 of Sports Goods (the adjacent row in the sorted DataFrame).

In [15]:
# --- Temporal Decomposition ---
master_df['Year']  = (master_df['Date_YYYYMM'] // 100).astype(int)
master_df['Month'] = (master_df['Date_YYYYMM'] % 100).astype(int)

# Cyclic encoding keeps January and December adjacent in feature space
# (a linear month column would wrongly treat them as far apart)
master_df['Month_Sin'] = np.sin(2 * np.pi * master_df['Month'] / 12)
master_df['Month_Cos'] = np.cos(2 * np.pi * master_df['Month'] / 12)

print('Temporal features created: Year, Month, Month_Sin, Month_Cos')

Temporal features created: Year, Month, Month_Sin, Month_Cos


In [16]:
# --- Lag Features (grouped by commodity) ---
for lag in [1, 3, 6]:
    master_df[f'Lag_{lag}M'] = (
        master_df.groupby('HS_Code')['Export_Value_USD'].shift(lag)
    )

lag_nulls = master_df[['Lag_1M', 'Lag_3M', 'Lag_6M']].isnull().sum().to_dict()
print('Lag features created: Lag_1M, Lag_3M, Lag_6M')
print(f'NaN counts (expected — early months per commodity): {lag_nulls}')

Lag features created: Lag_1M, Lag_3M, Lag_6M
NaN counts (expected — early months per commodity): {'Lag_1M': 10, 'Lag_3M': 30, 'Lag_6M': 60}


In [17]:
# --- Rolling Average Features (grouped by commodity) ---
# shift(1) inside the window prevents the current month from being in its own average
# min_periods=window ensures NaN until a full window of real values is available
for window in [3, 6]:
    master_df[f'Rolling_{window}M_Avg'] = (
        master_df.groupby('HS_Code')['Export_Value_USD']
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=window).mean())
    )

roll_nulls = master_df[['Rolling_3M_Avg', 'Rolling_6M_Avg']].isnull().sum().to_dict()
print('Rolling features created: Rolling_3M_Avg, Rolling_6M_Avg')
print(f'NaN counts (expected): {roll_nulls}')

ALL_ENG_COLS = ['Lag_1M', 'Lag_3M', 'Lag_6M', 'Rolling_3M_Avg', 'Rolling_6M_Avg']
nan_rows = master_df[ALL_ENG_COLS].isnull().any(axis=1).sum()
print(f'\nTotal rows with any NaN in engineered features: {nan_rows}')
print(f'({nan_rows // len(COMMODITY_FILES)} months per commodity — will be dropped in next step)')

Rolling features created: Rolling_3M_Avg, Rolling_6M_Avg
NaN counts (expected): {'Rolling_3M_Avg': 30, 'Rolling_6M_Avg': 60}

Total rows with any NaN in engineered features: 60
(6 months per commodity — will be dropped in next step)


---
## Section 8 — Drop NaN Rows (Option A: Clean Truncation)

The first 6 months of each commodity have incomplete lag and rolling features — there is no historical data before the series starts.  
We drop these rows entirely rather than filling them with artificial values.

- **Rows dropped:** 6 months × 10 commodities = **60 rows**
- **Final dataset:** 10 × 186 months = **1,860 rows** — still over 15 years of real data per commodity.

In [18]:
LAG_ROLLING_COLS = ['Lag_1M', 'Lag_3M', 'Lag_6M', 'Rolling_3M_Avg', 'Rolling_6M_Avg']

rows_before = len(master_df)
master_df   = master_df.dropna(subset=LAG_ROLLING_COLS).reset_index(drop=True)
rows_after  = len(master_df)

print(f'Rows before drop  : {rows_before:,}')
print(f'Rows dropped      : {rows_before - rows_after:,}  ({(rows_before - rows_after) // len(COMMODITY_FILES)} per commodity)')
print(f'Rows after drop   : {rows_after:,}')
print()
print(f'Remaining nulls in engineered features: {master_df[LAG_ROLLING_COLS].isnull().sum().sum()}')
print()
print('Rows per commodity after NaN drop:')
print(master_df.groupby('HS_Code').size().rename('Row Count').to_string())

Rows before drop  : 1,920
Rows dropped      : 60  (6 per commodity)
Rows after drop   : 1,860

Remaining nulls in engineered features: 0

Rows per commodity after NaN drop:
HS_Code
1006    186
1207    186
2523    186
5205    186
6110    186
6203    186
6302    186
7403    186
9018    186
9506    186


---
## Section 9 — Final Cleanup, Validation & Save

In [19]:
# Convert HS_Code to string — required for LightGBM / CatBoost categorical handling
master_df['HS_Code'] = master_df['HS_Code'].astype(str)

# Enforce a clean, logical column order for the final CSV
# (identifiers → target → external drivers → temporal → lag → rolling)
FINAL_COLS = [
    'Date_YYYYMM',
    'HS_Code',
    'Commodity_Name',
    'Export_Value_USD',
    'USD_PKR_Close',
    'Brent_Oil_Avg',
    'US_Consumer_Confidence',
    'Year',
    'Month',
    'Month_Sin',
    'Month_Cos',
    'Lag_1M',
    'Lag_3M',
    'Lag_6M',
    'Rolling_3M_Avg',
    'Rolling_6M_Avg',
]

master_df = master_df[FINAL_COLS]

print(f'Final shape        : {master_df.shape}')
print(f'Total columns      : {len(FINAL_COLS)}')
print(f'HS_Code dtype      : {master_df["HS_Code"].dtype}')
print(f'Columns: {FINAL_COLS}')

Final shape        : (1860, 16)
Total columns      : 16
HS_Code dtype      : str
Columns: ['Date_YYYYMM', 'HS_Code', 'Commodity_Name', 'Export_Value_USD', 'USD_PKR_Close', 'Brent_Oil_Avg', 'US_Consumer_Confidence', 'Year', 'Month', 'Month_Sin', 'Month_Cos', 'Lag_1M', 'Lag_3M', 'Lag_6M', 'Rolling_3M_Avg', 'Rolling_6M_Avg']


In [20]:
print('=' * 65)
print('  MASTER DATASET — FINAL VALIDATION REPORT')
print('=' * 65)
print(f'  Shape             : {master_df.shape}')
print(f'  Date range        : {master_df["Date_YYYYMM"].min()}  to  {master_df["Date_YYYYMM"].max()}')
print(f'  Commodities       : {master_df["HS_Code"].nunique()}  |  {sorted(master_df["HS_Code"].unique())}')
print(f'  Total null values : {master_df.isnull().sum().sum()}')
print()

print('--- Null Count per Column ---')
print(master_df.isnull().sum().to_string())

print()
print('--- Export_Value_USD Descriptive Statistics ---')
print(master_df['Export_Value_USD'].describe())

print()
print('--- Mean Export Value per Commodity (USD) ---')
commodity_means = (
    master_df.groupby('HS_Code')
    .agg(Commodity=('Commodity_Name', 'first'), Mean_USD=('Export_Value_USD', 'mean'))
    .sort_values('Mean_USD', ascending=False)
)
commodity_means['Mean_USD'] = commodity_means['Mean_USD'].apply(lambda x: f'${x:,.0f}')
print(commodity_means.to_string())

  MASTER DATASET — FINAL VALIDATION REPORT
  Shape             : (1860, 16)
  Date range        : 201007  to  202512
  Commodities       : 10  |  ['1006', '1207', '2523', '5205', '6110', '6203', '6302', '7403', '9018', '9506']
  Total null values : 0

--- Null Count per Column ---
Date_YYYYMM               0
HS_Code                   0
Commodity_Name            0
Export_Value_USD          0
USD_PKR_Close             0
Brent_Oil_Avg             0
US_Consumer_Confidence    0
Year                      0
Month                     0
Month_Sin                 0
Month_Cos                 0
Lag_1M                    0
Lag_3M                    0
Lag_6M                    0
Rolling_3M_Avg            0
Rolling_6M_Avg            0

--- Export_Value_USD Descriptive Statistics ---
count         1,860.00
mean     89,226,409.18
std      98,435,021.56
min               0.00
25%      18,008,439.63
50%      38,800,435.92
75%     143,944,428.02
max     518,469,801.71
Name: Export_Value_USD, dtype: float6

In [21]:
print('First 5 rows of the Master Dataset:')
master_df.head()

First 5 rows of the Master Dataset:


,Date_YYYYMM,HS_Code,Commodity_Name,Export_Value_USD,USD_PKR_Close,Brent_Oil_Avg,US_Consumer_Confidence,Year,Month,Month_Sin,Month_Cos,Lag_1M,Lag_3M,Lag_6M,Rolling_3M_Avg,Rolling_6M_Avg
0,201007,1006,Rice,"217,788,746.05",85.20,75.51,67.80,2010,7,-0.50,-0.87,"200,437,802.51","189,097,493.31","235,850,504.30","186,807,063.65","209,725,535.00"
1,201008,1006,Rice,"128,217,021.36",84.18,77.11,68.90,2010,8,-0.87,-0.50,"217,788,746.05","170,885,895.13","206,046,142.98","196,370,814.56","206,715,241.96"
2,201009,1006,Rice,"127,197,178.00",86.15,78.49,68.20,2010,9,-1.00,-0.00,"128,217,021.36","200,437,802.51","256,035,371.79","182,147,856.64","193,743,721.69"
3,201010,1006,Rice,"182,419,465.70",85.80,83.54,67.70,2010,10,-0.87,0.50,"127,197,178.00","217,788,746.05","189,097,493.31","157,734,315.14","172,270,689.39"
4,201011,1006,Rice,"146,964,729.76",85.70,86.16,71.60,2010,11,-0.50,0.87,"182,419,465.70","128,217,021.36","170,885,895.13","145,944,555.02","171,157,684.79"


In [22]:
output_path = OUTPUT_DIR / 'Master_FYP_Dataset.csv'
master_df.to_csv(output_path, index=False)

file_size_kb = output_path.stat().st_size / 1024

print('Master dataset saved successfully.')
print(f'Path      : {output_path.resolve()}')
print(f'File size : {file_size_kb:.1f} KB')
print(f'Rows      : {len(master_df):,}')
print(f'Columns   : {len(master_df.columns)}')

Master dataset saved successfully.
Path      : C:\Users\Talha Abbasi\Desktop\PECDF\Data\Master_FYP_Dataset.csv
File size : 518.1 KB
Rows      : 1,860
Columns   : 16
